# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a Croissant dataset using the `mlcroissant` library and references all fields, columns, and record sets by their `@id`.

### Dataset Source
The dataset source is available via Croissant schema at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print dataset name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets by @id
print('Record Sets:')
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    recset = dataset.record_sets[rs_id]
    print(f"- {rs_id}: {getattr(recset, 'name', '')}")

# Display fields and columns for each record set
for rs_id in record_sets:
    recset = dataset.record_sets[rs_id]
    print(f"\nRecord Set {rs_id} ({getattr(recset, 'name', '')}):")
    if hasattr(recset, 'fields'):
        print('  Fields:')
        for field in recset.fields:
            print(f"    - {field['@id']}: {field.get('name', '')} ({field.get('dataType', '')})")
    if hasattr(recset, 'columns'):
        print('  Columns:')
        for col in recset.columns:
            print(f"    - {col['@id']}: {col.get('name', '')} ({col.get('dataType', '')})")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. Record set and field `@id` values are used for referencing.

In [ ]:
# Extract and load data from each record set
import pprint
dataframes = {}

# List all record set @ids
record_sets = list(dataset.record_sets.keys())
print(f"Available record sets: {record_sets}")

# Load each record set by @id (common pattern is one main record set)
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nColumns for record set {record_set_id}:")
    pprint.pprint(df.columns.tolist())
    print(df.head())

# For further analysis, select the main record set by @id
# Edit this if you wish to select a different record set
main_record_set_id = record_sets[0] if record_sets else None

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, and grouping. Demonstrated using a numeric field by its `@id`.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Columns in {main_record_set_id}:", df.columns.tolist())

    # Identify a numeric field (column) by @id or string; adjust as needed for this dataset
    # Here we try to auto-detect a numeric column
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    if numeric_field is None:
        print("No numeric field auto-detected. Please set `numeric_field` to a numeric column @id.")
    else:
        print(f"Using numeric field: {numeric_field}")

        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 0

        # Filter records with numeric_field > threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field (e.g., categorical/label field)
        group_field = None
        for col in df.columns:
            if col != numeric_field and pd.api.types.is_object_dtype(df[col]):
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped means of {numeric_field} by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields with Matplotlib/Seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field is not None:
    # Histogram for the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by group_field if available
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load a protein abundance dataset from its Croissant schema, explore its structure, extract records by referencing entities via their `@id`, analyze and visualize the data, and perform basic EDA. This approach ensures reproducibility and precision through schema-level references, facilitating robust FAIR data workflows.